# Proyecto 1 — Monitoreo transaccional: detectar lo que el orden revela

**Universidad del Valle · Deep Learning 2026 · Kevin Recinos**

Integrantes: _(pendiente)_

**Pregunta central.** ¿El orden de las transacciones aporta información que las variables
agregadas no capturan, bajo qué condiciones y cuánto vale esa información en quetzales?

**Ruta de datos elegida: A — generador sintético propio.** Se eligió sobre la Ruta B porque
permite controlar qué mecanismo de fraude depende del orden y cuál no. Ese control es lo que
convierte la comparación A vs B en evidencia: si el modelo secuencial gana solo donde el orden
importa por construcción, la conclusión es verificable y no una correlación afortunada.

---

### Mapa del notebook

| Bloque | Contenido | Evidencia del informe |
|---|---|---|
| 0 | Configuración y entorno | Reproducibilidad |
| 1 | Generador, secuencias, partición temporal, antifuga | 1 — Integridad de datos |
| 2 | Modelo A (sin orden) y Modelo B (secuencial) | 2 — Comparación común |
| 3 | Permutación controlada y segunda prueba | 3 — Valor del orden |
| 4 | Apuesta C: hipótesis, control, veredicto | 4 — Apuesta del equipo |
| 5 | Umbral, costo y recomendación | 5 y 6 — Decisión y límites |

## Bloque 0 — Configuración y entorno

Una sola semilla gobierna generación de datos, partición, inicialización de pesos y barajado
de lotes. Las constantes de diseño viven en `src/config.py` y se declaran **antes** de ver
cualquier dato; el notebook no las redefine.

In [1]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore", category=FutureWarning)

from src import config
from src.utils import fijar_semillas, hash_df, dispositivo, asegurar_directorios, resumen_entorno

asegurar_directorios()
SEMILLA = fijar_semillas()
DISPOSITIVO = dispositivo()

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 9})

print(f"semilla global   : {SEMILLA}")
print(f"dispositivo      : {DISPOSITIVO}")
print(f"metrica principal: {config.METRICA_PRINCIPAL}  (la exactitud NO se reporta como principal)")
print(f"costos           : FN = Q{config.COSTO_FN:,}  |  FP = Q{config.COSTO_FP:,}")

semilla global   : 2026
dispositivo      : cpu
metrica principal: auc_pr  (la exactitud NO se reporta como principal)
costos           : FN = Q4,200  |  FP = Q180


In [2]:
# Versiones exactas del entorno de ejecucion (van tambien al README).
resumen_entorno()

,componente,version
0,python,3.11.9
1,plataforma,Windows-10-10.0.26200-SP0
2,numpy,2.4.6
3,pandas,2.3.3
4,sklearn,1.9.0
5,torch,2.13.0+cpu
6,matplotlib,3.11.0


## Bloque 1 — Datos, secuencias y protocolo temporal

> **Evidencia 1 del informe: integridad de datos.**
> Origen, tamaño, tasa de fraude, construcción de secuencias, partición temporal y
> controles contra fuga de información.

### 1.1 Generador sintético — mecanismos de fraude  *(T2)*

Cada tarjeta tiene un perfil propio: frecuencia de uso, monto típico, hora habitual y una lista
de comercios frecuentes. Sobre ese tráfico normal se inyectan episodios.

| Mecanismo | Patrón en palabras sencillas | ¿Depende del orden? |
|---|---|---|
| **F1 — escalada** | Varias compras pequeñas de prueba, en montos crecientes, y enseguida el golpe grande. | **Sí.** Los mismos montos en otro orden son una compra legítima. |
| **F2 — ráfaga** | Muchos cargos en pocos minutos, en comercios distintos y desde el extranjero. | Parcialmente: pesa más la densidad temporal que la secuencia exacta. |
| **F3 — atípico aislado** | Un solo cargo muy por encima del monto habitual de esa tarjeta. | **No.** Es el control del experimento. |

**El control que hace honesta la comparación.** Junto a cada escalada se generan dos
*sesiones legítimas*: mismos montos, misma ventana de tiempo, mismos comercios nuevos, mismo
canal — pero en orden **no creciente**. Sin ellas, "varias compras chicas y una grande" sería
separable con solo mirar el promedio y el máximo de la ventana, y el orden no tendría nada que
aportar. Con ellas, la única diferencia sistemática entre fraude y no fraude en F1 es la
secuencia.

**Dónde esperamos que el modelo falle.** Las primeras sondas de una escalada están etiquetadas
como fraude —la tarjeta ya está comprometida— pero son indistinguibles de una compra pequeña
cualquiera: en ese punto todavía no existe historia que delate el patrón. Ningún modelo,
secuencial o no, debería detectarlas, y esperamos ver ese piso en el desglose por posición
dentro del episodio.

In [3]:
from src.generador import generar

tx = generar()
huella = hash_df(tx)

print(f"transacciones : {len(tx):,}")
print(f"tarjetas      : {tx.id_tarjeta.nunique():,}")
print(f"ventana       : {tx.timestamp.min():%Y-%m-%d} a {tx.timestamp.max():%Y-%m-%d}")
print(f"tasa de fraude: {tx.es_fraude.mean()*100:.3f} %  (objetivo {config.TASA_FRAUDE_OBJETIVO*100:.3f} %)")
print(f"huella        : {huella}")

assert huella == hash_df(generar()), "el generador no es reproducible con la misma semilla"
assert huella != hash_df(generar(seed=config.SEED + 1)), "cambiar la semilla no cambio los datos"
print("\nreproducibilidad verificada: misma semilla -> misma huella, otra semilla -> otra huella")

transacciones : 324,603
tarjetas      : 5,991
ventana       : 2025-01-01 a 2025-04-30
tasa de fraude: 1.197 %  (objetivo 1.200 %)
huella        : 6087b55f499a77dd



reproducibilidad verificada: misma semilla -> misma huella, otra semilla -> otra huella


**Composición del conjunto.** Las sesiones legítimas no son fraude: son el control de F1.

In [4]:
episodios = tx.id_episodio.where(tx.id_episodio >= 0)
composicion = (tx.assign(episodio=episodios)
                 .groupby("mecanismo", observed=True)
                 .agg(transacciones=("es_fraude", "size"),
                      episodios=("episodio", "nunique"),
                      etiqueta=("es_fraude", "max"),
                      monto_mediano=("monto", "median"))
                 .sort_values("transacciones", ascending=False))
composicion["% del total"] = (composicion.transacciones / len(tx) * 100).round(3)
composicion

,transacciones,episodios,etiqueta,monto_mediano,% del total
mecanismo,,,,,
legitimo,317186,0,0,71.23,97.715
sesion_legitima,3531,702,0,30.60,1.088
F1_escalada,1766,351,1,29.49,0.544
F2_rafaga,1341,130,1,79.63,0.413
F3_atipico,779,779,1,1706.78,0.240


**Una escalada y su control, lado a lado.** Mismos ingredientes, distinto orden.

In [5]:
def episodio_ejemplo(mecanismo, golpe_al_final):
    """Primer episodio del mecanismo cuyo monto mayor cae (o no) en la ultima posicion."""
    cols = ["timestamp", "monto", "id_comercio", "canal", "es_fraude"]
    for eid in tx.loc[tx.mecanismo == mecanismo, "id_episodio"].unique():
        ep = tx.loc[tx.id_episodio == eid, cols]
        if (ep.monto.values.argmax() == len(ep) - 1) == golpe_al_final:
            return ep.reset_index(drop=True)

print("F1_escalada (fraude): montos crecientes que rematan en el golpe")
print(episodio_ejemplo("F1_escalada", golpe_al_final=True).to_string(index=False))
print()
print("sesion_legitima (control): mismos ingredientes, el golpe no va al final")
print(episodio_ejemplo("sesion_legitima", golpe_al_final=False).to_string(index=False))

F1_escalada (fraude): montos crecientes que rematan en el golpe
          timestamp  monto  id_comercio     canal  es_fraude
2025-03-28 20:54:00  13.62          114 ecommerce          1
2025-03-28 20:57:00  38.65           95 ecommerce          1
2025-03-28 21:03:00  40.71         1109 ecommerce          1
2025-03-28 21:04:00  41.13          100 ecommerce          1
2025-03-28 21:08:00 539.13          304 ecommerce          1

sesion_legitima (control): mismos ingredientes, el golpe no va al final
          timestamp  monto  id_comercio     canal  es_fraude
2025-03-10 12:01:00  42.51          287 ecommerce          0
2025-03-10 12:03:00 768.77          753 ecommerce          0
2025-03-10 12:07:00   7.70          792 ecommerce          0
2025-03-10 12:14:00  15.68          463 ecommerce          0
2025-03-10 12:19:00  28.14          938 ecommerce          0
2025-03-10 12:23:00  29.53          623 ecommerce          0


**El control funciona.** Los agregados de ventana —conteo, suma, promedio, máximo, diversidad de
comercios, duración— son estadísticamente iguales entre F1 y su control. Lo único que separa a
los dos grupos es la monotonía de la secuencia.

In [6]:
comparables = tx[tx.mecanismo.isin(["F1_escalada", "sesion_legitima"])]
por_episodio = comparables.groupby(["mecanismo", "id_episodio"], observed=True).agg(
    eventos=("monto", "size"),
    suma=("monto", "sum"),
    promedio=("monto", "mean"),
    maximo=("monto", "max"),
    comercios=("id_comercio", "nunique"),
    duracion_min=("timestamp", lambda s: (s.max() - s.min()).total_seconds() / 60),
)
agregados = por_episodio.groupby("mecanismo", observed=True).mean().round(2)

creciente = (comparables.groupby(["mecanismo", "id_episodio"], observed=True)
             .monto.apply(lambda s: bool(np.all(np.diff(s.values) > 0)))
             .groupby("mecanismo", observed=True).mean() * 100).round(1)
agregados["% estrictamente creciente"] = creciente
agregados

,eventos,suma,promedio,maximo,comercios,duracion_min,% estrictamente creciente
mecanismo,,,,,,,
F1_escalada,5.03,1302.55,265.03,1203.14,5.03,18.23,99.7
sesion_legitima,5.03,1327.35,270.40,1225.12,5.03,18.08,0.0


La única señal que queda al alcance de un modelo sin orden es *dónde cae el monto grande*: en
una escalada siempre es el último evento, mientras que en el control cae al final solo una de
cada cinco veces. Ese solapamiento del 20 % es el techo que la línea base no puede superar en F1,
y es exactamente lo que el modelo secuencial debería poder romper.

In [7]:
posicion_golpe = (comparables.groupby(["mecanismo", "id_episodio"], observed=True)
                  .monto.apply(lambda s: bool(s.values.argmax() == len(s) - 1))
                  .groupby("mecanismo", observed=True).mean() * 100).round(1)
posicion_golpe.rename("% con el golpe al final del episodio").to_frame()

,% con el golpe al final del episodio
mecanismo,
F1_escalada,100.0
sesion_legitima,20.4


**Persistencia.** El CSV es un artefacto derivado: se regenera con la semilla, no se versiona.

In [8]:
ruta_datos = config.DIR_DATOS / "transacciones.csv"
tx.to_csv(ruta_datos, index=False)
print(f"{ruta_datos.name}: {ruta_datos.stat().st_size / 1e6:.1f} MB")

transacciones.csv: 24.7 MB


### 1.2 Construcción de secuencias  *(T3)*

Cada ejemplo es la historia de una tarjeta hasta el evento actual **inclusive**, y se predice
si ese evento es fraudulento. Ese es el horizonte que enfrenta un motor antifraude real:
autorizar o bloquear la transacción que está ocurriendo ahora.

In [9]:
# TODO (T3): construccion de secuencias y particion temporal

### 1.3 Partición temporal y controles antifuga  *(T3, T4)*

Lo más antiguo entrena, lo intermedio valida, lo más reciente prueba. **El conjunto de prueba
se mira una sola vez**, después de fijar arquitectura, hiperparámetros y umbral.

In [10]:
# TODO (T4): agregadas causales + escalado ajustado SOLO en train

## Bloque 2 — Núcleo comparable: A contra B

> **Evidencia 2 del informe: comparación común.**
> Mismos datos, misma partición, mismo horizonte. AUC-PR y, en el umbral elegido,
> precisión, exhaustividad y F1.

### 2.1 Modelo A — línea base sin orden  *(T5)*

In [11]:
# TODO (T5): modelo A sobre variables agregadas

### 2.2 Modelo B — modelo secuencial  *(T6)*

In [12]:
# TODO (T6): modelo B sobre eventos ordenados

### 2.3 Comparación A vs B  *(T7)*

In [13]:
# TODO (T7): curva PR conjunta, AUC-PR, y precision/recall/F1 en el umbral de validacion

## Bloque 3 — ¿De verdad usó el orden?

> **Evidencia 3 del informe: valor del orden.**
> Una mejora de métricas no demuestra por sí sola que el modelo leyó la secuencia.
> Estas dos pruebas intentan refutar nuestra propia conclusión.

### 3.1 Prueba obligatoria — permutación controlada  *(T8)*

In [14]:
# TODO (T8): barajar el orden DENTRO de cada secuencia, sin tocar eventos ni agregadas

### 3.2 Segunda prueba — desempeño por mecanismo de fraude  *(T9)*

In [15]:
# TODO (T9): AUC-PR de A y B desglosado por F1 / F2 / F3

## Bloque 4 — Apuesta del equipo

> **Evidencia 4 del informe: apuesta del equipo.**
> Hipótesis previa, control experimental, métrica de éxito y veredicto — aunque falle.

**La hipótesis se escribe y se commitea ANTES de entrenar C y ANTES de tocar el conjunto de prueba.**

### 4.1 Hipótesis previa  *(T10)*

> Creemos que ______ mejorará ______ porque ______. Lo consideraremos útil si ______.

_(pendiente — T10)_

In [16]:
# TODO (T11): entrenar C y su control, y emitir veredicto explicito

## Bloque 5 — Umbral, costo y recomendación

> **Evidencias 5 y 6 del informe: decisión económica, recomendación y límites.**
> Un fraude no detectado cuesta Q4,200; bloquear una transacción legítima cuesta Q180.
> El umbral se elige barriendo **validación** y se aplica una sola vez a prueba.

In [17]:
# TODO (T12): barrido de umbral por costo esperado y proyeccion mensual

## Matriz de evidencias

| Evidencia | Figura o tabla | Conclusión | Limitación |
|---|---|---|---|
| 1 — Integridad de datos | _(pendiente)_ | | |
| 2 — Comparación común A vs B | _(pendiente)_ | | |
| 3 — Valor del orden | _(pendiente)_ | | |
| 4 — Apuesta del equipo | _(pendiente)_ | | |
| 5 — Decisión económica | _(pendiente)_ | | |
| 6 — Recomendación y límites | _(pendiente)_ | | |